# ANALISIS DE SENTIMIENTOS APLICANDO FINETUNING
# AUTOR : CÉSAR MAYTA


# 1. Instalar librerías

In [ ]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


# 2. Cargar dataset en español

In [ ]:
from datasets import load_dataset

dataset = load_dataset("pyupeu/social-media-peruvian-sentiment")

dataset

README.md:   0%|          | 0.00/857 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/990k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/306k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/255k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9336 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2918 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2335 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 9336
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 2918
    })
    test: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 2335
    })
})

# 3. Ver columnas y ejemplos

In [ ]:
print(dataset)

print(dataset["train"].features)

print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 9336
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 2918
    })
    test: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 2335
    })
})
{'text': Value('string'), 'label': Value('int64'), 'label_name': Value('string')}
{'text': 'Salió para Aser farándula ese chato, solo quiere llamar la atención 😂 q trabaje bago , aragán', 'label': 0, 'label_name': 'negative'}


# 4. Crear mapas de etiquetas

In [ ]:
from collections import Counter

print(Counter(dataset["train"]["label_name"]))
print(Counter(dataset["train"]["label"]))

Counter({'negative': 4265, 'positive': 3033, 'neutral': 2038})
Counter({0: 4265, 2: 3033, 1: 2038})


In [ ]:
labels = sorted(set(dataset["train"]["label_name"]))

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)
print(id2label)

{'negative': 0, 'neutral': 1, 'positive': 2}
{0: 'negative', 1: 'neutral', 2: 'positive'}


# 5. Cargar modelo base en español

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "dccuchile/bert-base-spanish-wwm-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/486k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

# 6. Tokenizar dataset

In [ ]:
def tokenizar(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenizar,
    batched=True,
    remove_columns=["text", "label_name"]
)

tokenized_dataset

Map:   0%|          | 0/9336 [00:00<?, ? examples/s]

Map:   0%|          | 0/2918 [00:00<?, ? examples/s]

Map:   0%|          | 0/2335 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9336
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2918
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2335
    })
})

In [ ]:
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# 7. Configurar métricas

In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

# 8. Configurar entrenamiento

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./modelo_sentimiento_peruano",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# 9. Entrenar

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.679609,0.720147,0.693626,0.669137
2,0.455023,0.783612,0.699794,0.699562


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2334, training_loss=0.6282218019474033, metrics={'train_runtime': 513.1989, 'train_samples_per_second': 36.384, 'train_steps_per_second': 4.548, 'total_flos': 1228213433954304.0, 'train_loss': 0.6282218019474033, 'epoch': 2.0})

# 10. Evaluar en test

In [ ]:
trainer.evaluate(tokenized_dataset["test"])

{'eval_loss': 0.8311824798583984,
 'eval_accuracy': 0.6882226980728051,
 'eval_f1': 0.68651638380952,
 'eval_runtime': 20.9088,
 'eval_samples_per_second': 111.675,
 'eval_steps_per_second': 13.965,
 'epoch': 2.0}

# 11. Guardar modelo

In [ ]:
trainer.save_model("./modelo_sentimiento_peruano")
tokenizer.save_pretrained("./modelo_sentimiento_peruano")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./modelo_sentimiento_peruano/tokenizer_config.json',
 './modelo_sentimiento_peruano/tokenizer.json')

# 12. Probar modelo entrenado

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./modelo_sentimiento_peruano",
    tokenizer="./modelo_sentimiento_peruano"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
frases = [
    "Este curso estuvo bravazo, aprendí un montón.",
    "Qué pésimo servicio, nunca más compro aquí.",
    "La atención fue normal, nada fuera de lo común.",
    "Me encantó la clase, estuvo muy clara.",
    "Ese producto no pasa nada, muy malo.",
    "Más o menos nomás, esperaba algo mejor."
]

classifier(frases)

[{'label': 'positive', 'score': 0.980541467666626},
 {'label': 'negative', 'score': 0.40874484181404114},
 {'label': 'neutral', 'score': 0.5793347358703613},
 {'label': 'positive', 'score': 0.9766636490821838},
 {'label': 'negative', 'score': 0.9818857312202454},
 {'label': 'neutral', 'score': 0.7038698196411133}]

# 12 publicamos el modelo

In [ ]:
trainer.push_to_hub('analisis-sentimiento-peruano')

print(f"¡El modelo ha sido re-publicado exitosamente en Hugging Face con el nombre sentimiento peruano!")